# Multi-view reconstruction method benchmark

## 0. Import

In [ ]:
%matplotlib inline
import openalea.phenomenal.data as phm_data
import openalea.phenomenal.object as phm_obj
import openalea.phenomenal.multi_view_reconstruction as phm_mvr
import openalea.phenomenal.display as phm_display
import openalea.phenomenal.display.notebook as phm_display_notebook
from openalea.phenotyping_data.fetch import fetch_all_data

## 1. Prerequisites

### 1.1 Load data

In [ ]:
plant_number = 2  # Available : 1, 2, 3, 4 or 5
data_dir = fetch_all_data(f"plant_{plant_number}")

In [ ]:
bin_images = phm_data.bin_images(data_dir)
calibrations = phm_data.calibrations(data_dir)

phm_display.show_images(
    list(bin_images["side"].values()) + list(bin_images["top"].values())
)

In [ ]:
image_views = dict()
for id_camera in bin_images:
    for angle in bin_images[id_camera]:
        name = f'{id_camera}_{angle}'
        projection = calibrations[id_camera].get_projection(angle)
        image_views[name] = phm_obj.ImageView(
                bin_images[id_camera][angle],
                projection
            )

## 4 Comparison of reconstruction methods

In [ ]:
import time
times, voxels, metrics = {}, {}, {}
voxels_size = 16  # mm

### 4.1 Rigid reconstruction (fully consistent)

In [ ]:
method = 'rigid'
error_tolerance = 0
start = time.time()
voxel_grid = phm_mvr.reconstruction_3d(
    image_views, voxels_size=voxels_size, error_tolerance=error_tolerance, clear_outside='side')
end = time.time()
times[method] = end - start
voxels[method] = voxel_grid
metrics[method] = phm_mvr.reconstruction_metrics(voxel_grid, image_views)

### 4.2 tolerant reconstruction

In [ ]:
method = 'tolerant'
error_tolerance = 2
start = time.time()
voxel_grid = phm_mvr.reconstruction_3d(
    image_views, voxels_size=voxels_size, error_tolerance=error_tolerance, clear_outside='side')
end = time.time()
times[method] = end - start
voxels[method] = voxel_grid
metrics[method] = phm_mvr.reconstruction_metrics(voxel_grid, image_views)

### 4.3 multi-tolerant reconstruction

In [ ]:
method = 'multi-tolerant'
error_tolerance = -6
start = time.time()
voxel_grid = phm_mvr.reconstruction_3d(
    image_views, voxels_size=voxels_size, error_tolerance=error_tolerance, clear_outside='side')
end = time.time()
times[method] = end - start
voxels[method] = voxel_grid
metrics[method] = phm_mvr.reconstruction_metrics(voxel_grid, image_views)

### 4.4 rigid + 'best view' reconstruction

In [ ]:
def find_best_angle(bin_side_images):
    max_len = 0
    max_angle = None

    for angle in bin_side_images:
        x_pos, y_pos, x_len, y_len = cv2.boundingRect(
            cv2.findNonZero(bin_side_images[angle])
        )

        if x_len > max_len:
            max_len = x_len
            max_angle = angle

    return max_angle

In [ ]:
method = 'best_angle'
error_tolerance = 0
start = time.time()
best_angle = find_best_angle(bin_images["side"])
image_ref = f'side_{best_angle}'
voxel_grid = phm_mvr.reconstruction_3d_neighbours(
    image_views, voxels_size=voxels_size, error_tolerance=error_tolerance, clear_outside='side', reference_views=image_ref
)
end = time.time()
times[method] = end - start
voxels[method] = voxel_grid
metrics[method] = phm_mvr.reconstruction_metrics(voxel_grid, image_views)

## Visual comparison

In [ ]:
binded=phm_obj.bind_grids(list(voxels.values()), -1300)

In [ ]:
phm_display_notebook.show_voxel_grid(binded, size=1)

In [ ]:
## Computation times

In [ ]:
import matplotlib.pyplot as plt
data = times
labels = list(data.keys())
values = list(data.values())

plt.figure()
plt.bar(labels, values)

plt.xlabel("Method")
plt.ylabel("Value")
plt.title("Comparison of Computation times")

plt.show()

In [ ]:
data = metrics
plt.figure()

for method, vals in data.items():
    plt.scatter(vals['recall'], vals['precision'], s=80)
    plt.text(vals['recall'] + 0.002, vals['precision'], method)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall Trade-off")
plt.xlim(0, 1.05)
plt.ylim(0, 1.05)
plt.grid(True)

plt.show()

In [ ]:
import numpy
methods = list(data.keys())
x = numpy.arange(len(methods))
width = 0.35

iou_vals = [data[m]["IoU"] for m in methods]
dice_vals = [data[m]["Dice"] for m in methods]

plt.figure()

plt.bar(x - width/2, iou_vals, width, label="IoU")
plt.bar(x + width/2, dice_vals, width, label="Dice")

plt.xticks(x, methods, rotation=20)
plt.ylabel("Score")
plt.title("Reconstruction Quality (IoU vs Dice)")
plt.ylim(0, 1)
plt.legend()

plt.show()